# Lean-43 — La borne de couverture C(24,14,4) ≥ 20, certifiée en Lean

**Navigation** : [Index](README.md) | [<< Lean-42 L'indépendance arithmétique, certifiée](Lean-42-Independance-Arithmetique-BusyBeaver-Lean.ipynb) | [English version](Lean-43-Borne-Couverture-C24-14-4-Lean_en.ipynb)

**Lake** : [`covering_lean/`](covering_lean/README.md) — consommateur épinglé du paquet `covering` de [jamesyc/covering](https://github.com/jamesyc/covering).

**Source** : papier *A structural lower bound for C(24,14,4)* (Apache-2.0), archivé au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness/2026 - Chang - A structural lower bound for C(24,14,4) (github jamesyc-covering, Apache-2.0, Lean formalise).pdf`. Issue de série : #27477, jalon 3.

**Objectifs d'apprentissage**

- lire un **lac consommateur** Lean : où vit le paquet amont, à quel commit, avec quel toolchain ;
- distinguer une **borne inférieure** prouvée d'un **nombre de couverture** déterminé — la confusion la plus coûteuse de ce chapitre ;
- lire un **audit d'axiomes** (`#print axioms`) et le confronter à la liste que l'amont s'autorise lui-même ;
- relier une borne combinatoire à la **récurrence de Schönheim**, et voir ce qu'elle propage et ce qu'elle ne prouve pas.

**Prérequis** : notions de combinatoire (familles de blocs, designs de couverture) et de Lean 4 (modules, `lake`, `#print axioms`). Le carnet [Lean-42](Lean-42-Independance-Arithmetique-BusyBeaver-Lean.ipynb) a posé le motif du lac consommateur ; celui-ci enchaîne sur un exemple de combinatoire.

**Durée estimée** : 45 minutes de lecture, plus le temps d'un build Lean (de l'ordre de quelques minutes sur un lac déjà construit).

## Plan

1. **Le lac consommateur** : où vit le paquet amont, à quel commit, avec quel toolchain.
2. **Le pin** : relire le manifeste et vérifier que la révision tient.
3. **Le build** : construire le témoin, lire ce que `lake` a réellement fait.
4. **L'audit d'axiomes** : `#print axioms` sur les deux théorèmes, et le verdict mécanique.
5. **Les nombres du papier** : bornes, tableau du dépôt amont, propagation par Schönheim.
6. **Lecture** : ce que « C(24,14,4) ≥ 20 » établit, et ce qu'il ne dit pas.
7. **Exercices** : trois, à écrire dans les cellules stub.

## 1. Le lac consommateur

Le lac `covering_lean` ne définit **rien** : il épingle le paquet `covering` du dépôt amont et importe ses théorèmes. La clause du `lakefile.lean` est directe :

```lean
require «covering» from git
  "https://github.com/jamesyc/covering.git" @ "e28962cff844f155ca5a5cfb22563083369b490e"
```

Aucun fork, aucune copie, aucune adaptation : Lake clone le dépôt amont au commit annoncé et charge le paquet tel qu'il est publié.

L'amont déclare **trois** bibliothèques, et elles ne jouent pas le même rôle :

| Bibliothèque | Rôle | Contenu |
|---|---|---|
| `Covering` | la preuve | les modules qui construisent le raisonnement, jusqu'à `Covering/Main.lean` |
| `Challenge` | l'artefact d'audit | les deux énoncés **avec `sorry`** — la question posée, pas la réponse |
| `Solution` | la réponse | les mêmes énoncés, prouvés, pour la soumission au registre Palomar |

Le consommateur n'importe que `Covering`. Il ne touche ni `Challenge` (les `sorry` y sont volontaires : c'est le cahier des charges) ni `Solution` (redondante avec `Covering.Main`).

Le toolchain du lac est **v4.35.0-rc3**, celui du paquet amont, qui épingle Mathlib au même tag. Ce n'est pas un oubli : la série compte maintenant trois lakes à trois toolchains (`formal_logic_lean` en 4.33.1, `busybeaver_lean` en v4.35.0, `covering_lean` en v4.35.0-rc3). Un lake par toolchain est la forme prévue par Lake ; les trois vivent côte à côte, sans dépendance entre eux.

La cellule suivante localise le lac, calcule le chemin WSL, et compare les empreintes SHA-256 des fichiers de build entre le dépôt et le lake effectivement utilisé : un build réussi dans ce lac vaut pour le dépôt **seulement** si les quatre empreintes coïncident.

In [1]:
import hashlib
import os
import pathlib
import subprocess

NB_DIR = pathlib.Path.cwd()
REPO_LAKE = NB_DIR / "covering_lean"
PIN_ATTENDU = "e28962cff844f155ca5a5cfb22563083369b490e"


def wslpath(p):
    # Chemin Windows -> chemin WSL, calcule par WSL lui-meme.
    r = subprocess.run(["wsl", "-e", "wslpath", "-a", str(p).replace(chr(92), "/")],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    return r.stdout.strip()


def wsl_sh(commande, tentatives=3):
    # Execute une commande shell dans WSL et rend sa sortie standard.
    # WSL peut refuser une invocation transitoirement (machine virtuelle occupee
    # par un build borne) : on rejoue, et on leve si l'echec est reel plutot
    # que de rendre une sortie vide que l'on croirait lire.
    dernier = None
    for _ in range(tentatives):
        r = subprocess.run(["wsl", "-e", "bash", "-lc", commande],
                           capture_output=True, text=True, encoding="utf-8", errors="replace")
        if r.returncode == 0:
            return r.stdout
        dernier = r
    raise RuntimeError(f"wsl a echoue (rc={dernier.returncode}) : {commande}\n"
                       f"{dernier.stderr.strip()}")


LAKE_OVERRIDE = os.environ.get("COVERING_LAKE_WSL")
LAKE_DIR = LAKE_OVERRIDE or wslpath(REPO_LAKE)
LEAN_EXEC = wslpath(NB_DIR.parents[2] / "scripts" / "lean" / "lean_exec.py")


def lean_exec(commande, timeout=900):
    # Lance `commande` dans le lake via l'organe borne ; rend (code, sortie, journal).
    r = subprocess.run(
        ["wsl", "-e", "bash", "-lc",
         f"cd {LAKE_DIR} && python3 {LEAN_EXEC} run --timeout {timeout} -- {commande}"],
        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout + 120)
    journal = [l for l in r.stderr.splitlines() if l.startswith("[lean_exec]")]
    return r.returncode, r.stdout, journal


FICHIERS_BUILD = ["lean-toolchain", "lakefile.lean", "lake-manifest.json", "CoveringConsumer.lean"]

# Tout est accumule puis imprime en UNE fois : un seul flush stdout, donc une
# seule sortie commitee -- le jumeau EN ne peut pas decouper autrement.
lignes = ["lake utilise : " + ("copie (COVERING_LAKE_WSL)" if LAKE_OVERRIDE
                               else "covering_lean du depot")]
cote_wsl = wsl_sh(f"cd {LAKE_DIR} && sha256sum " + " ".join(FICHIERS_BUILD)).splitlines()
empreintes_wsl = {l.split()[1]: l.split()[0] for l in cote_wsl if l.strip()}
identiques = 0
for nom in FICHIERS_BUILD:
    h_depot = hashlib.sha256((REPO_LAKE / nom).read_bytes()).hexdigest()
    h_lake = empreintes_wsl.get(nom, "absent")
    identiques += h_depot == h_lake
    lignes.append(f"  {nom:22s} depot {h_depot[:12]}  lake {h_lake[:12]}  "
                  f"{'identique' if h_depot == h_lake else 'DIFFERENT'}")
lignes.append(f"{identiques}/{len(FICHIERS_BUILD)} fichiers identiques"
              + ("  (comparaison triviale : meme dossier)" if not LAKE_OVERRIDE else ""))

# Le manifeste peut annoncer un pin ; la tete du clone reellement construit le
# prouve. Les deux sont lus : le manifeste a la cellule suivante, la tete ici.
tete = wsl_sh(f"git -C {LAKE_DIR}/.lake/packages/covering rev-parse HEAD").strip()
lignes.append("tete du clone amont : " + tete)
lignes.append("pin tenu (clone)    : " + str(tete == PIN_ATTENDU))
lignes.append("toolchain           : " + (REPO_LAKE / "lean-toolchain").read_text().strip())
print("\n".join(lignes))

lake utilise : copie (COVERING_LAKE_WSL)
  lean-toolchain         depot bc84812c9448  lake bc84812c9448  identique
  lakefile.lean          depot c67a35e583ad  lake c67a35e583ad  identique
  lake-manifest.json     depot 1871390ce742  lake 1871390ce742  identique
  CoveringConsumer.lean  depot 32764c9bb682  lake 32764c9bb682  identique
4/4 fichiers identiques
tete du clone amont : e28962cff844f155ca5a5cfb22563083369b490e
pin tenu (clone)    : True
toolchain           : leanprover/lean4:v4.35.0-rc3


### Lecture de la mise en place

Quatre lignes, une par fichier de build : chacune affiche l'empreinte côté dépôt et l'empreinte côté lake. Toutes doivent porter `identique` — sinon le lake construit est un autre objet que celui du dépôt (copie oubliée, état intermédiaire). Quand le lake est une copie préparée à l'avance (variable `COVERING_LAKE_WSL`), la comparaison est réelle ; sur le lake du dépôt elle est trivialement vraie, et la cellule le dit plutôt que de le laisser croire.

Deux lignes ferment le point aveugle du manifeste : `git rev-parse HEAD` sur le clone amont rend la révision **réellement extraite**, et `pin tenu (clone)` doit valoir `True`. Un manifeste peut annoncer un pin que le clone n'a pas ; la tête du clone ne le peut pas. Le toolchain affiché est celui que `lake` utilisera — `v4.35.0-rc3`.

## 2. Le pin, relu dans le manifeste

`lake update` a résolu la clause `from git` en une révision inscrite dans `lake-manifest.json`. Ce fichier est le contrat : toute construction ultérieure part de ces révisions, pas de la branche amont.

La cellule lit le manifeste et cherche le paquet `covering`. On attend le commit `e28962cf…` — celui sur lequel l'amont publie son paquet, son audit et son papier. Un manifeste qui porterait une autre révision serait une dérive silencieuse : le build réussirait, mais sur un autre contenu.

In [2]:
# Le manifeste Lake fige le commit exact du paquet : c'est la preuve du pin.
import json

manifeste = json.loads((REPO_LAKE / "lake-manifest.json").read_text(encoding="utf-8"))
# `covering` est la dependance directe ; les autres paquets viennent de ses transitives.
paquet = next(p for p in manifeste["packages"] if p["name"] == "covering")
print("paquet      :", paquet["name"])
print("rev         :", paquet["rev"])
print("url         :", paquet["url"])
print("pin tenu    :", paquet["rev"] == PIN_ATTENDU)
print("sous-dir lu :", paquet.get("subDir"))
print("dependances :", ", ".join(sorted(p["name"] for p in manifeste["packages"]
                                       if p["name"] != "covering")))

paquet      : covering
rev         : e28962cff844f155ca5a5cfb22563083369b490e
url         : https://github.com/jamesyc/covering.git
pin tenu    : True
sous-dir lu : None
dependances : Cli, LeanSearchClient, Qq, aesop, batteries, importGraph, mathlib, plausible, proofwidgets


### Lecture du pin

Quatre valeurs à confronter : `rev` doit égaler le pin attendu (`pin tenu : True`), `url` doit désigner `jamesyc/covering`, `sous-dir lu` doit valoir `None` — le paquet est à la racine du dépôt amont, contrairement au paquet `PA196` du carnet Lean-42, logé dans un sous-répertoire — et la liste des dépendances doit être celle que le paquet amont déclare. Ces dépendances — Mathlib, puis `aesop`, `batteries`, `Qq`, `Cli`, `plausible`, `proofwidgets`, `import-graph`, `LeanSearchClient` — sont résolues par le paquet amont lui-même : ce lake ne les re-déclare pas, donc il ne peut pas les faire diverger.

Mathlib est ici en `v4.35.0-rc3`, la révision que l'amont épingle : la garantie de cohérence entre le paquet et sa bibliothèque de base tient sans notre intervention.

## 3. Le build

La cellule lance `lake build` dans le lake, par l'organe `scripts/lean/lean_exec.py` : cap machine-wide, confinement de l'arbre de processus, budget mesuré. C'est le même organe que les autres carnets Lean de la série ; aucun `lake` n'est lancé à la main.

Sur un lake neuf, deux commandes précèdent : `lake update` (résolution du pin et écriture du manifeste) puis `lake exe cache get` (oléans Mathlib de la révision amont). Sans le cache, Mathlib se compilerait depuis les sources — plusieurs heures pour un contenu que l'amont publie déjà. Le carnet les tient pour acquises : son objet est le build du témoin, pas la préparation du lake.

Un premier build paie ensuite la compilation du paquet amont — un peu plus de deux mille modules — et les suivants sont incrémentaux. Le nombre de tâches affiché par `lake` est donc un état du cache, pas une propriété du résultat : c'est pourquoi la cellule ne retient que la ligne de conclusion.

In [3]:
# Build du temoin : incremental une fois les oleans en place.
code_retour, sortie, journal = lean_exec("lake build", timeout=5400)
# Seule la ligne de conclusion est retenue : le detail des taches (et leur
# duree) depend de l'etat du cache, pas du resultat.
utiles = [l for l in sortie.splitlines() if l.startswith("Build completed successfully")]
print("\n".join(utiles[-1:]))
print("build reussi :", code_retour == 0)
# Le temps de build varie d'un run a l'autre : normalise, sinon le jumeau EN
# ne peut pas porter la meme sortie (garde twin-parity).
import re
print("\n".join(re.sub(r"duration=[0-9.]+s", "duration=mesuree", l) for l in journal))
print("code de retour :", code_retour)

Build completed successfully (2180 jobs).
build reussi : True
[lean_exec] ok exit=0 duration=mesuree orphans=0
[lean_exec] child_exit_code: 0
code de retour : 0


### Lecture du build

Le **code de retour** doit être nul : un `lake build` qui échoue rend un code non nul et des lignes d'erreur. Le **journal de l'organe** porte la mesure d'admission et la postcondition « zéro descendant orphelin » ; sa durée est normalisée, car elle dépend de la machine et non du résultat.

La ligne de conclusion — `Build completed successfully` — est celle qui compte : elle signifie que l'ensemble des cibles a été construit, le paquet amont comme le module consommateur. Un build incrémental peut n'afficher aucune tâche : c'est le signe que les `.olean` sont à jour, pas que rien n'a été vérifié.

Le paquet amont publie ses propres vérifications : le module `Solution` porte les `#print axioms` de ses deux théorèmes, et le fichier `comparator.json` la liste des axiomes que l'amont s'autorise. La section suivante confronte notre audit à cette liste, plutôt qu'à une liste imaginée sur place.

## 4. L'audit d'axiomes

Lean permet de demander, pour chaque théorème, la liste des axiomes dont sa preuve dépend. C'est la mesure qui distingue une preuve complète d'une preuve qui s'appuie sur `sorry` (`sorryAx`) ou sur l'évaluation native (`Lean.ofReduceBool`, via `native_decide`).

Le module `CoveringConsumer.lean` du lac écrit deux `#print axioms` : `Covering.covering_24_14_4` et `Covering.covering_25_15_5`. La cellule ré-élabore le module avec `lake env lean` — les imports sont résolus par le lake — et lit les lignes d'axiomes.

Le verdict est ensuite mécanique, pas déclaratif : la cellule suivante compare l'ensemble cité à la liste blanche `propext`, `Classical.choice`, `Quot.sound`, à la liste interdite `sorryAx`, `Lean.ofReduceBool`, `Lean.trustCompiler`, **et** à la liste que l'amont publie lui-même dans `comparator.json`.

In [4]:
# Audit d'axiomes : `lake env lean` re-elabore le module et imprime chaque
# ligne `#print axioms` ecrite dans CoveringConsumer.lean.
code_retour, sortie, journal = lean_exec("lake env lean CoveringConsumer.lean", timeout=1800)
lignes_axiomes = [l.strip() for l in sortie.splitlines() if "axioms" in l.lower()]
print("\n".join(lignes_axiomes))
print("code de retour :", code_retour)

'Covering.covering_24_14_4' depends on axioms: [propext, Classical.choice, Quot.sound]
'Covering.covering_25_15_5' depends on axioms: [propext, Classical.choice, Quot.sound]
code de retour : 0


### Lecture des lignes d'axiomes

Deux lignes, une par théorème, et la même liste dans les deux cas : `propext`, `Classical.choice`, `Quot.sound`. Ce sont les trois axiomes du noyau de Lean et de Mathlib — l'extensionalité propositionnelle, le choix classique, les quotients. Aucun n'est un trou de preuve : `sorryAx` en serait un, et il ne figure pas dans la liste.

La cellule suivante ne se contente pas de cette lecture : elle confronte l'ensemble mesuré à la liste que l'amont publie dans `comparator.json`, c'est-à-dire à ce que l'amont lui-même déclare acceptable pour sa soumission. C'est un contrôle croisé, pas une relecture de la même liste deux fois.

In [5]:
AUTORISES = {"propext", "Classical.choice", "Quot.sound"}
INTERDITS = {"sorryAx", "Lean.ofReduceBool", "Lean.trustCompiler"}


def axiomes_cites(lignes):
    # Extrait l'ensemble des axiomes cites par les lignes `#print axioms`.
    trouves = set()
    for ligne in lignes:
        if "depends on axioms:" in ligne:
            liste = ligne.split("depends on axioms:")[-1].strip().strip("[]")
            trouves |= {a.strip() for a in liste.split(",") if a.strip()}
        # « does not depend on any axioms » : ensemble vide, rien a ajouter.
    return trouves


cites = axiomes_cites(lignes_axiomes)
print("axiomes cites :", sorted(cites) if cites else "(aucun)")
print("hors liste blanche :", sorted(cites - AUTORISES) or "(aucun)")
print("interdits presents :", sorted(cites & INTERDITS) or "(aucun)")
print("audit concluant :", cites <= AUTORISES and not (cites & INTERDITS))

# Controle croise : la liste que l'amont publie lui-meme pour sa soumission.
comparateur_texte = wsl_sh(f"cat {LAKE_DIR}/.lake/packages/covering/comparator.json")
if not comparateur_texte.strip():
    raise RuntimeError("comparator.json vide : la lecture amont a echoue")
comparateur = json.loads(comparateur_texte)
autorises_amont = set(comparateur["permitted_axioms"])
print("liste de l'amont :", sorted(autorises_amont))
print("concordance avec l'amont :", cites <= autorises_amont)
print("theoremes declares :", " | ".join(comparateur["theorem_names"]))

axiomes cites : ['Classical.choice', 'Quot.sound', 'propext']
hors liste blanche : (aucun)
interdits presents : (aucun)
audit concluant : True
liste de l'amont : ['Classical.choice', 'Quot.sound', 'propext']
concordance avec l'amont : True
theoremes declares : Covering.Palomar.covering_24_14_4_lower_bound | Covering.Palomar.covering_25_15_5_lower_bound


### Lecture de l'audit

Trois verdicts se lisent ici, et ils sont distincts.

`audit concluant : True` est le seul résultat acceptable côté noyau : l'ensemble cité est un sous-ensemble de la liste blanche, et aucun axiome interdit n'apparaît. Un `sorryAx` signalerait une preuve laissée vide chez l'amont ; un `Lean.ofReduceBool` signalerait un appel à `native_decide`, dont la confiance repose sur le compilateur plutôt que sur le noyau.

`concordance avec l'amont : True` est un contrôle **indépendant** : la liste de référence n'est pas celle qu'on a écrite à la main au-dessus, c'est celle du fichier `comparator.json` publié par l'amont. Un écart entre les deux signalerait que l'amont s'autorise davantage que le noyau de Lean n'exige — ou que le fichier du dépôt et le paquet construit ne sont pas le même objet.

`theoremes declares` nomme les deux énoncés que l'amont soumet au registre : `Covering.Palomar.covering_24_14_4_lower_bound` et `Covering.Palomar.covering_25_15_5_lower_bound`. Ce sont les mêmes énoncés, exprimés dans la formulation `Finset` usuelle, que ceux que le module `Covering/Main.lean` prouve et que le carnet lit ci-dessous.

## 5. Les nombres du papier

Le résultat est un théorème, mais son énoncé cite des nombres : 24 points, des blocs de 14, des 4-sous-ensembles à couvrir, et la borne 20. La cellule lit ces nombres **dans le paquet consommé**, à la révision épinglée — le tableau des bornes est parsé depuis le `README.md` amont et les constantes sont relevées dans `Covering/Main.lean` — puis elle calcule la propagation par la récurrence de Schönheim.

Elle ne re-prouve rien : la preuve est celle de l'amont, consommée par le lake. Confronter un énoncé à sa donnée est une hygiène de lecture, pas une revendication de résultat.

In [6]:
# Les nombres, lus dans le paquet consomme a la revision epinglee : le tableau
# des bornes est parse depuis le README amont, pas recopie depuis le texte.
import re

texte_amont = wsl_sh(f"cat {LAKE_DIR}/.lake/packages/covering/README.md")
if not texte_amont.strip():
    raise RuntimeError("README amont vide : la lecture amont a echoue")
bornes = {}
for ligne in texte_amont.splitlines():
    m = re.match(r"\|\s*C\((\d+),\s*(\d+),\s*(\d+)\)\s*\|\s*(\d+)\s*\|\s*\*\*(\d+)\*\*\s*\|\s*(\d+)\s*\|",
                 ligne)
    if m:
        v, k, t, avant, apres, meilleur = (int(x) for x in m.groups())
        bornes[(v, k, t)] = (avant, apres, meilleur)

print("bornes lues dans le tableau amont :")
for (v, k, t), (avant, apres, meilleur) in sorted(bornes.items()):
    print(f"  C({v},{k},{t}) : borne precedente {avant} -> borne prouvee {apres}"
          f" | meilleure couverture connue {meilleur}")

# Les constantes des enonces, relevees dans le module de l'amont.
main_lean = wsl_sh(f"cat {LAKE_DIR}/.lake/packages/covering/Covering/Main.lean")
for motif, libelle in [
    (r"Finset \(Fin 24\)", "point set Fin 24"),
    (r"card = 14", "blocs de 14"),
    (r"card = 4 ", "sous-ensembles de 4"),
    ("20 ≤ ", "borne 20"),
    (r"Finset \(Fin 25\)", "point set Fin 25"),
    (r"card = 15", "blocs de 15"),
    (r"card = 5 ", "sous-ensembles de 5"),
    ("34 ≤ ", "borne 34"),
]:
    print(f"enonce contient {libelle:22s} :", bool(re.search(motif, main_lean)))

# La recurrence de Schonheim : C(v,k,t) >= ceil(v/k * C(v-1,k-1,t-1)).
def schoenheim(v, k, borne_interieure):
    # Plus petit entier superieur ou egal a (v * borne_interieure) / k.
    return -(-v * borne_interieure // k)


propages = [(26, 16, 6, 34), (27, 17, 7, 56), (28, 18, 8, 89)]
lus = re.search(r"improves C\(26,16,6\) to (\d+), C\(27,17,7\) to (\d+) "
                r"and C\(28,18,8\) to (\d+)", texte_amont)
lus = [int(x) for x in lus.groups()] if lus else []
print("propagation par Schonheim (25,15,5) -> 34 :", schoenheim(25, 15, 20))
for (v, k, t, base), annonce in zip(propages, lus + [None] * 3):
    calcule = schoenheim(v, k, base)
    print(f"  C({v},{k},{t}) : {base} -> {calcule} | amont annonce {annonce}"
          + (" | concordance : " + str(calcule == annonce) if annonce else ""))

bornes lues dans le tableau amont :
  C(24,14,4) : borne precedente 19 -> borne prouvee 20 | meilleure couverture connue 23
  C(25,15,5) : borne precedente 32 -> borne prouvee 34 | meilleure couverture connue 42
enonce contient point set Fin 24       : True
enonce contient blocs de 14            : True
enonce contient sous-ensembles de 4    : True
enonce contient borne 20               : True
enonce contient point set Fin 25       : True
enonce contient blocs de 15            : True
enonce contient sous-ensembles de 5    : True
enonce contient borne 34               : True
propagation par Schonheim (25,15,5) -> 34 : 34
  C(26,16,6) : 34 -> 56 | amont annonce 56 | concordance : True
  C(27,17,7) : 56 -> 89 | amont annonce 89 | concordance : True
  C(28,18,8) : 89 -> 139 | amont annonce 139 | concordance : True


### Lecture des nombres

Trois lectures, trois portée différentes.

**Le tableau amont.** Les bornes y sont en trois colonnes : la borne inférieure précédemment prouvée, la borne inférieure prouvée par le paquet, et la meilleure couverture **connue**. Pour C(24,14,4) : 19 → 20, la meilleure construction connue en demande 23 blocs. Pour C(25,15,5) : 32 → 34, meilleure construction connue en demande 42 blocs. Le gain est de **un** bloc sur la première, de **deux** sur la seconde — et c'est exactement ce que la propagation par Schönheim produit, comme on le voit plus bas.

**Les constantes des énoncés.** Les huit lignes `enonce contient …` doivent toutes porter `True` : c'est la relecture de l'énoncé dans sa source, non dans un commentaire. Le module de l'amont dit bien `Fin 24`, des blocs de 14, des 4-sous-ensembles, une borne `20 ≤`, puis `Fin 25`, des blocs de 15, des 5-sous-ensembles, une borne `34 ≤`.

**La propagation.** La récurrence de Schönheim énonce qu'une couverture de `v` points par des blocs de `k` exige au moins `⌈v/k × C(v−1,k−1,t−1)⌉` blocs. Appliquée à C(25,15,5) avec la borne 20, elle donne `⌈25 × 20 / 15⌉ = 34`. Appliquée encore, elle donne C(26,16,6) → 56, C(27,17,7) → 89, C(28,18,8) → 139 — les valeurs que le dépôt amont annonce lui aussi. Chaque `concordance : True` confronte notre calcul à la phrase de l'amont.

Ces trois derniers nombres ne sont **pas** prouvés par le paquet : ils sont la conséquence arithmétique de la borne de Schönheim, appliquée par nous. Le paquet ne les formalise pas — l'amont le dit explicitement.

## 6. Lecture : ce que « C(24,14,4) ≥ 20 » dit, et ce qu'il ne dit pas

Les deux théorèmes se répondent, et il vaut la peine de les séparer :

| Théorème | Énoncé | Ce qu'il établit |
|---|---|---|
| `covering_24_14_4` | `20 ≤ 𝒯.card` | toute famille de blocs de 14 couvrant les 4-sous-ensembles de 24 points a **au moins** 20 blocs |
| `covering_25_15_5` | `34 ≤ 𝒯.card` | toute famille de blocs de 15 couvrant les 5-sous-ensembles de 25 points a **au moins** 34 blocs |

Le second est le corollaire du premier : il passe par les dérivées à un point et la borne de Schönheim. Il ne coûte pas un second raisonnement.

Ce que cela **ne dit pas** :

- **ni** que `C(24,14,4) = 20`. Une borne inférieure de 20 laisse ouvertes les valeurs 20, 21, 22 et 23 ; la meilleure construction connue en demande 23. L'espace entre les deux est exactement l'objet de la question ouverte.
- **ni** que le nombre de couverture est déterminé. Le dépôt amont est explicite : il établit des bornes, il ne détermine aucun des deux nombres de couverture.
- **ni** que la borne 34 pour C(25,15,5) est optimale. La meilleure construction connue en demande 42 ; le second borne est de 32.

L'intérêt de la formalisation n'est pas le nombre — un point de plus sur une borne connue depuis des décennies de combinatoire. Il est dans la **traçabilité** : la chaîne de déductions (comptages de degrés et d'incidences, arguments de rang, résultats de rigidité, puis exclusion des configurations restantes) est écrite en Lean et vérifiée par le noyau. Une erreur de cas.base dans l'une de ces étapes ne passerait pas inaperçue, alors qu'elle passerait probablement inaperçue dans un article.

C'est le versant formel qui manquait au carnet [Complexity-08d](../../InformationCalcul/Complexity/Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb) : celui-ci présentait les bornes et annonçait leur branchement ; celui-ci les branche.

### Exercice 1 — Le prédicat d'audit

In [7]:
# Exercice 1 : ecrire le predicat d'audit qui refuserait un axiome interdit.
def audit_concluant(lignes):
    # TODO etudiant : renvoyer True si tous les axiomes cites sont dans AUTORISES.
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : audit_concluant(lignes_axiomes)")

Exercice a completer : audit_concluant(lignes_axiomes)


### Exercice 2 — La récurrence de Schönheim

In [8]:
# Exercice 2 : ecrire la borne de Schonheim C(v,k,t) >= ceil(v/k * borne interieure).
def borne_de_schonheim(v, k, borne_interieure):
    # TODO etudiant : renvoyer le plus petit entier superieur ou egal a
    # (v * borne_interieure) / k.
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : borne_de_schonheim(25, 15, 20)")

Exercice a completer : borne_de_schonheim(25, 15, 20)


### Exercice 3 — Borne inférieure et nombre de couverture

In [9]:
# Exercice 3 : distinguer une borne inferieure prouvee d'un nombre de couverture.
def classe_du_resultat(borne_inferieure, meilleure_couverture_connue):
    # TODO etudiant : renvoyer "borne inferieure" si le resultat ne donne qu'un
    # plancher, "nombre de couverture" s'il determine exactement C(v,k,t).
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : classe_du_resultat(20, 23)")

Exercice a completer : classe_du_resultat(20, 23)


## 7. Conclusion

- Le paquet `covering` est consommé **tel quel**, au commit épinglé, par la clause `from git` : aucun fork, aucune copie, aucune adaptation.
- Le build passe sous l'organe borné, et l'audit d'axiomes ne cite que `propext`, `Classical.choice`, `Quot.sound` — un ensemble **concordant** avec la liste que l'amont publie lui-même dans `comparator.json`. Aucun `sorryAx`, aucun `native_decide`.
- Le résultat est une **borne inférieure** : `C(24,14,4) ≥ 20` et `C(25,15,5) ≥ 34`. Les nombres de couverture eux-mêmes restent indéterminés, et le dépôt amont le dit.
- Les chiffres sont lus dans le paquet consommé, à la révision épinglée : le tableau des bornes, les constantes des énoncés, et la propagation de Schönheim recalculée puis confrontée aux valeurs annoncées par l'amont.

### Références

- Papier *A structural lower bound for C(24,14,4)* (Apache-2.0) : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Chang - A structural lower bound for C(24,14,4) (github jamesyc-covering, Apache-2.0, Lean formalise).pdf`
- Dépôt : [jamesyc/covering](https://github.com/jamesyc/covering), commit `e28962cff844f155ca5a5cfb22563083369b490e`
- Site compagnon : <https://jamesyc.com/covering/>
- Registre Palomar : `PALOMAR-2026-10-04-000003`
- Série : [Complexity-08d — Indépendance prouvée, Busy Beaver et coverings](../../InformationCalcul/Complexity/Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb)
- Issue de série : #27477 (jalon 3 — Greffe B)